# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AzlanFaisalRaj/flyrank-internship-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

The label `is_declining_label` is a yes/no outcome I already have for every row (derived from
`trend_direction`), and the Week-4 baseline ranks rows for a reviewer, evaluated at
precision@10 / precision@20. That is a "yes/no with an observed label, read as a ranking"
shape, so I start with **Logistic Regression** (readable, gives calibrated-ish probabilities
to rank by) and compare it against a **Random Forest** (stronger, but only worth keeping if it
actually beats the simpler model on the same metric). I am not reaching for Gradient Boosting
here — the toolkit note is clear that complexity should earn its place through the comparison,
not be added by default, and two models is enough to see whether more complexity even helps.


In [1]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("rows:", len(df), "| clients:", df["client_id"].nunique())
print("label balance:")
print(df["is_declining_label"].value_counts(normalize=True).round(3))


rows: 30000 | clients: 32
label balance:
is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I split by **`client_id`** (`GroupShuffleSplit`, 75/25), not row-by-row. Content items from the
same client share client-level patterns (industry, content ops cadence, how aggressively they
publish), so a random row split would leak client identity across train and test and make the
model look better than it really is. There is no usable timestamp column in this starter CSV
to build a time-aware split on (the only "time" signal, `trend_pct`/`trend_direction`, is the
label itself), so grouped-by-client is the honest split available here. I check for zero client
overlap between train and test before trusting anything downstream.


In [2]:
from sklearn.model_selection import GroupShuffleSplit

RANDOM_SEED = 42

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))
train, test = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

print("train rows:", len(train), "| test rows:", len(test))
print("train clients:", train["client_id"].nunique(), "| test clients:", test["client_id"].nunique())
print("client overlap (must be 0):", len(set(train["client_id"]) & set(test["client_id"])))
print("train base rate:", round(train["is_declining_label"].mean(), 3),
      "| test base rate:", round(test["is_declining_label"].mean(), 3))


train rows: 22885 | test rows: 7115
train clients: 24 | test clients: 8
client overlap (must be 0): 0
train base rate: 0.55 | test base rate: 0.517


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Features drop three things: the two IDs (`content_id`, `client_id` — grouping only, never
features, per the data skill), the label-derived columns (`trend_direction`, `trend_pct`), and
every `*_last_30d` / `*_prev_30d` column. That last exclusion matches the Week-4 baseline: the
label is built directly from `impressions_last_30d` vs `impressions_prev_30d`, so any column in
that window is functionally the label in disguise, not a real signal. 34 columns remain (23
numeric, 11 categorical).

I recompute the Week-4 rule baseline **on this same test split** (it was originally scored on
all 30,000 rows with no split) so the comparison is apples-to-apples: same rows, same
precision@10 / precision@20 metric, plus the base rate as the honest floor.


In [3]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

window_cols = [c for c in df.columns if c.endswith("_last_30d") or c.endswith("_prev_30d")]
drop_cols = {"content_id", "client_id", "trend_direction", "trend_pct", "is_declining_label"} | set(window_cols)
feature_cols = [c for c in df.columns if c not in drop_cols]
num_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df[c])]
cat_cols = [c for c in feature_cols if not pd.api.types.is_numeric_dtype(df[c])]
print("dropped window cols:", window_cols)
print("n features:", len(feature_cols), "| numeric:", len(num_cols), "| categorical:", len(cat_cols))

X_train, y_train = train[feature_cols], train["is_declining_label"]
X_test, y_test = test[feature_cols], test["is_declining_label"]

cat_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
pre_rf = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", cat_pipe, cat_cols),
])
pre_logreg = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num_cols),
    ("cat", cat_pipe, cat_cols),
])

logreg = Pipeline([("pre", pre_logreg), ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_SEED))])
rf = Pipeline([("pre", pre_rf), ("clf", RandomForestClassifier(
    n_estimators=300, max_depth=8, random_state=RANDOM_SEED, n_jobs=-1))])

logreg.fit(X_train, y_train)
rf.fit(X_train, y_train)

def precision_at_k(y_true, scores, k):
    order = np.argsort(-scores)
    return float(np.asarray(y_true)[order][:k].mean())

results = {}
for name, model in [("logreg", logreg), ("random_forest", rf)]:
    proba = model.predict_proba(X_test)[:, 1]
    results[name] = {
        "p@10": precision_at_k(y_test.values, proba, 10),
        "p@20": precision_at_k(y_test.values, proba, 20),
        "p@50": precision_at_k(y_test.values, proba, 50),
        "roc_auc": roc_auc_score(y_test, proba),
    }

# Week-4 rule baseline, recomputed on THIS test split for a fair comparison
expected_ctr_by_tier = train[train["impressions_90d"] >= 300].groupby("position_tier")["ctr"].median()
test_b = test.copy()
test_b["expected_ctr"] = test_b["position_tier"].map(expected_ctr_by_tier)
stale = test_b["freshness_tier"] == "91-180"
visible = test_b["impressions_90d"] >= 300
ctr_gap = test_b["ctr"] < test_b["expected_ctr"]
test_b["score"] = (stale.astype(int) * visible.astype(int) * test_b["impressions_90d"] * (1 + ctr_gap.astype(int))).round(1)
baseline_ranked = test_b.sort_values("score", ascending=False)
rank_scores = np.arange(len(baseline_ranked), 0, -1)  # already sorted, so this preserves rank order
results["baseline_rule"] = {
    "p@10": precision_at_k(baseline_ranked["is_declining_label"].values, rank_scores, 10),
    "p@20": precision_at_k(baseline_ranked["is_declining_label"].values, rank_scores, 20),
    "p@50": precision_at_k(baseline_ranked["is_declining_label"].values, rank_scores, 50),
    "roc_auc": np.nan,
}
base_rate = test["is_declining_label"].mean()
results["base_rate_floor"] = {"p@10": base_rate, "p@20": base_rate, "p@50": base_rate, "roc_auc": np.nan}

comparison_table = pd.DataFrame(results).T.round(3)
print(f"=== model vs baseline, test split n={len(test)} ===")
comparison_table


dropped window cols: ['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']
n features: 34 | numeric: 23 | categorical: 11


=== model vs baseline, test split n=7115 ===


,p@10,p@20,p@50,roc_auc
logreg,0.800,0.700,0.660,0.575
random_forest,0.500,0.500,0.500,0.597
baseline_rule,0.300,0.300,0.340,NaN
base_rate_floor,0.517,0.517,0.517,NaN


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Both models beat the Week-4 rule baseline (0.30 / 0.30 / 0.34) and the base rate (0.517) on
precision@10, so an observed model is a real step up from the hand-written rule for a top-of-queue
reviewer. But the two models disagree on **how** they win: Logistic Regression is stronger at
precision@10 (0.80) and precision@20 (0.70), while Random Forest has the better overall ROC AUC
(0.597 vs 0.575) but only matches the base rate at precision@10/20/50. That is the finding, not
a footnote — for this task the reviewer only ever looks at the top of the queue, so precision@K
is what matters, and the simpler model is the better pick here despite the Random Forest's
higher AUC. This matches the toolkit's own warning: don't reward complexity that doesn't earn
its place on the metric that matters.

Permutation importance (scored on ROC AUC) says the Random Forest leans hardest on
`days_with_impressions` and `impressions_90d` — both visibility/traffic-volume signals — then
`avg_position`, `ctr`, and `position_tier`. That is sane: visible, ranked pages are exactly
where a real decline shows up first. None of the top features are suspiciously perfect
(no single feature separates the classes cleanly), which is a mild anti-leakage signal on top
of the column exclusions from Section 3.

Looking at concrete errors: false positives (predicted declining, actually stable/up) cluster
in `page_3_5` and `page_1` tiers with mid-range probabilities (0.69-0.85) — content that looks
stale and underperforming on paper but didn't actually decline, plausibly because the 91-180
freshness window is a noisy proxy rather than a guarantee. False negatives (missed real
declines) sit at lower, less confident probabilities (0.34-0.44) in `striking` and `page_3_5`
tiers — borderline cases the model correctly treats as uncertain rather than confidently wrong,
which is a healthier failure mode than confident misses.


In [4]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf, X_test, y_test, n_repeats=5, random_state=RANDOM_SEED, n_jobs=-1, scoring="roc_auc")
imp_df = pd.DataFrame({"feature": feature_cols, "importance": perm.importances_mean}).sort_values(
    "importance", ascending=False)
print("Top 8 permutation importances (random forest, scored on ROC AUC):")
print(imp_df.head(8).to_string(index=False))

test_out = test.copy()
test_out["rf_proba"] = rf.predict_proba(X_test)[:, 1]
test_out["rf_pred"] = (test_out["rf_proba"] >= 0.5).astype(int)
fp = test_out[(test_out["rf_pred"] == 1) & (test_out["is_declining_label"] == 0)]
fn = test_out[(test_out["rf_pred"] == 0) & (test_out["is_declining_label"] == 1)]

cols = ["content_id", "freshness_tier", "impressions_90d", "ctr", "position_tier", "rf_proba"]
print(f"\nfalse positives: {len(fp)} | false negatives: {len(fn)}")
print("\n3 false positives (predicted decline, actually stable/up):")
print(fp[cols].head(3).to_string(index=False))
print("\n3 false negatives (missed a real decline):")
print(fn[cols].head(3).to_string(index=False))


Top 8 permutation importances (random forest, scored on ROC AUC):
              feature  importance
days_with_impressions    0.030249
      impressions_90d    0.011946
         avg_position    0.004944
                  ctr    0.003387
        position_tier    0.003015
     content_age_days    0.002731
           clicks_90d    0.002711
          scroll_rate    0.001952

false positives: 2048 | false negatives: 978

3 false positives (predicted decline, actually stable/up):
          content_id freshness_tier  impressions_90d  ctr position_tier  rf_proba
content_a5a2fbc76336         91-180              307 0.00      page_3_5  0.845667
content_72c5c2d73e5a           0-30             2426 0.12      page_3_5  0.692002
content_55f75c034970           0-30             3998 0.03        page_1  0.751638

3 false negatives (missed a real decline):
          content_id freshness_tier  impressions_90d  ctr position_tier  rf_proba
content_a1fb4e703a9e           0-30            15320 0.05      page_

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.